In [1]:
# Module 5: Context Management & Multi-Agent Orchestration
# Lab: Research & Synthesis Pipeline

# Setup -- install dependencies (run once per session)
# !pip install -q claude-agent-sdk python-dotenv

In [2]:
# Import libraries
import os
import json
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from claude_agent_sdk import query, ClaudeAgentOptions

In [3]:
# Load API keys from .env file
load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print(f"Anthropic key (SDK): {'Yes' if ANTHROPIC_API_KEY else 'No'}")

Anthropic key (SDK): Yes


In [4]:
# Step 1 -- Define the Researcher sub-agent
# The Researcher only has WebSearch and WebFetch. No write access.
async def run_researcher(topic: str) -> str:
    """Gather research on a topic using web tools only."""
    options = ClaudeAgentOptions(
        allowed_tools=["WebSearch", "WebFetch"],
        model="claude-haiku-4-5-20251001",
    )
    prompt = f"""Research the topic '{topic}' and return concise findings.
You MUST call WebSearch first to find relevant information, then WebFetch to read details.
Return a bullet-point summary of the most important facts only."""
    result = ""
    async for message in query(prompt=prompt, options=options):
        if hasattr(message, 'content') and message.content:
            result = message.content
        if hasattr(message, 'result') and message.result:
            result = message.result
    return result

In [5]:
# Step 2 -- Define the Writer sub-agent
# The Writer only has the Edit tool. No web access.
async def run_writer(findings: str, template_path: str, output_path: str) -> str:
    """Write findings into the report template using Edit tool only."""
    options = ClaudeAgentOptions(
        allowed_tools=["Read", "Edit"],
        permission_mode="bypassPermissions",
        model="claude-haiku-4-5-20251001",
    )
    prompt = f"""Read the template at {template_path}, then write a completed
report to {output_path} using the Edit tool.

Findings to incorporate:
{findings}

Replace every placeholder in the template with real content.
Do NOT modify any other files."""
    result = ""
    async for message in query(prompt=prompt, options=options):
        if hasattr(message, 'content') and message.content:
            result = message.content
        if hasattr(message, 'result') and message.result:
            result = message.result
    return result

In [6]:
# Step 3 -- Define the Coordinator
# The Coordinator orchestrates the full pipeline: research -> write
async def run_coordinator(task: str, template_path: str, output_path: str) -> str:
    """Orchestrate research and writing phases."""
    print("[Coordinator] Starting research phase...")
    findings = await run_researcher(task)
    print(f"[Coordinator] Research complete. {len(findings)} chars gathered.")

    print("[Coordinator] Starting writing phase...")
    report = await run_writer(findings, template_path, output_path)
    print("[Coordinator] Report written.")

    return report

In [7]:
# Step 4 -- Execute the pipeline
# Set target paths and run the full orchestration
# Each sub-agent gets its own fresh context window
TEMPLATE_PATH = "data/report_template.md"
OUTPUT_PATH = "data/completed_report.md"
TASK = "Quantum Computing"

result = await run_coordinator(TASK, TEMPLATE_PATH, OUTPUT_PATH)
print("\n--- Final Report ---\n")
print(result)

[Coordinator] Starting research phase...
[Coordinator] Research complete. 1 chars gathered.
[Coordinator] Starting writing phase...
[Coordinator] Report written.

--- Final Report ---

[TextBlock(text="The permission dialog should appear on your screen for writing to the data directory. Please approve the permission, and I'll complete the report creation.\n\nAlternatively, if you'd like me to proceed without waiting, I can create the file using a Bash command. Would you like me to do that instead?")]


In [8]:
# Step 5 -- Verify the output
# Read the completed report to verify the Writer filled in the template
report_file = Path(OUTPUT_PATH)
if report_file.exists():
    print("--- Completed Report ---")
    print(report_file.read_text())
else:
    print("Report not found.")

Report not found.
